In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

In [10]:
class GCN(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [11]:
def train(model, data, train_mask, optimizer, criterion, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()
        print(f'Epoch: {epoch+1}, Loss: {loss.item()}')

In [12]:
def evaluate(model, data, val_mask):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        loss = criterion(logits[val_mask], data.y[val_mask])
        _, pred = logits.max(dim=1)
        correct = int(pred[val_mask].eq(data.y[val_mask]).sum().item())
        acc = correct / val_mask.sum().item()
        print(f'Validation Loss: {loss.item()}, Accuracy: {acc}')

In [13]:
def test(model, data):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        _, pred = logits.max(dim=1)
        correct = int(pred[data.test_mask].eq(data.y[data.test_mask]).sum().item())
        acc = correct / data.test_mask.sum().item()
        print(f'Test Accuracy: {acc}')

In [17]:
# Step 2: Load and preprocess the dataset
dataset = Planetoid(root='../res/datasets/cora', name='Cora')
data = dataset[0]

# Step 3: Split the dataset into training and validation sets
train_mask, val_mask = train_test_split(data.y, test_size=0.1, stratify=data.y)


# Step 5: Define the loss function and optimizer
model = GCN(data.num_features, 16, dataset.num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train the model
train(model, data, train_mask, optimizer, criterion, num_epochs=200)

# Evaluate on validation set
evaluate(model, data, val_mask)

# Evaluate on test set
test(model, data)

Epoch: 1, Loss: 1.9108086824417114
Epoch: 2, Loss: 1.6602649688720703
Epoch: 3, Loss: 1.4401483535766602
Epoch: 4, Loss: 1.2175285816192627
Epoch: 5, Loss: 1.007815957069397
Epoch: 6, Loss: 0.8122708201408386
Epoch: 7, Loss: 0.6420666575431824
Epoch: 8, Loss: 0.49805253744125366
Epoch: 9, Loss: 0.3800838589668274
Epoch: 10, Loss: 0.2856958508491516
Epoch: 11, Loss: 0.21193620562553406
Epoch: 12, Loss: 0.15568383038043976
Epoch: 13, Loss: 0.11351409554481506
Epoch: 14, Loss: 0.08229213207960129
Epoch: 15, Loss: 0.059446025639772415
Epoch: 16, Loss: 0.042905546724796295
Epoch: 17, Loss: 0.031047437340021133
Epoch: 18, Loss: 0.022610900923609734
Epoch: 19, Loss: 0.016640087589621544
Epoch: 20, Loss: 0.012407040223479271
Epoch: 21, Loss: 0.009392925538122654
Epoch: 22, Loss: 0.007230543997138739
Epoch: 23, Loss: 0.005663156975060701
Epoch: 24, Loss: 0.004512788262218237
Epoch: 25, Loss: 0.003657870925962925
Epoch: 26, Loss: 0.003014049492776394
Epoch: 27, Loss: 0.0025213644839823246
Epoch: